# Full Data Collection — Bluesky

**Project:** *The Devil Wears Prada 2* — Social Media Analysis
**Platform:** Bluesky (AT Protocol via `atproto` SDK)

This notebook produces the complete dataset for the project in two phases.

The **first phase** collects posts matching a refined set of search queries informed by the pilot. Each query has a dedicated post limit calibrated to its observed yield and engagement: queries that saturated the pilot cap are widened, queries that proved weak or noisy are dropped or replaced, and several new queries are added to broaden recall.

The **second phase** reconstructs the conversation threads. For every root post that has at least one declared reply, the full thread is fetched and recursively flattened: every reply, at every depth, is recorded together with the URI of its parent post and the URI of the thread root. This produces the `replies_full` dataset used to build the conversation graph in the next notebook.

The notebook writes three deliverables to `data/raw/`: `posts_full`, `replies_full`, and an updated `query_summary_full`.


## 1. Setup

In [18]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from atproto import Client, models
from atproto_client.exceptions import (
    RequestException,
    InvokeTimeoutError,
    NetworkError,
)


In [19]:
with open("../credentials.json", "r") as f:
    credentials = json.load(f)

client = Client()
client.login(credentials["handle"], credentials["app_password"])
print("Logged in as:", credentials["handle"])


Logged in as: martapaniconi.bsky.social


## 2. Helper functions

The same primitives used in the pilot. `call_with_retries` honours rate-limit and timeout signals so that the long full-collection run can recover from transient errors without manual intervention.


In [20]:
def _parse_dt_utc(iso_str: str) -> datetime:
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _now_utc_ts() -> int:
    return int(datetime.now(timezone.utc).timestamp())


def call_with_retries(callable_fn, *args, max_retries: int = 10, **kwargs):
    for attempt in range(max_retries):
        try:
            return callable_fn(*args, **kwargs)

        except InvokeTimeoutError:
            wait_s = min(60, 2 ** attempt)
            print(f"[timeout] sleeping {wait_s}s")
            time.sleep(wait_s)
            continue

        except NetworkError:
            wait_s = min(60, 2 ** attempt)
            print(f"[network] sleeping {wait_s}s")
            time.sleep(wait_s)
            continue

        except RequestException as e:
            resp = getattr(e, "response", None)
            status = getattr(resp, "status_code", None)
            headers = getattr(resp, "headers", {}) or {}

            if status == 429:
                reset = headers.get("ratelimit-reset") or headers.get("RateLimit-Reset")
                retry_after = headers.get("retry-after") or headers.get("Retry-After")
                if reset:
                    wait_s = max(1, int(reset) - _now_utc_ts()) + 1
                elif retry_after:
                    wait_s = int(float(retry_after))
                else:
                    wait_s = min(60, 2 ** attempt)
                print(f"[429] rate limited, sleeping {wait_s}s")
                time.sleep(wait_s)
                continue

            wait_s = min(20, 2 ** attempt)
            print(f"[{status}] request failed, sleeping {wait_s}s")
            time.sleep(wait_s)

    raise RuntimeError("Too many retries / repeated failures.")


In [21]:
def extract_facets(record) -> dict:
    """Extract hashtags, mentions (DIDs), and external links from a post record."""
    hashtags, mentions, links = [], [], []

    for facet in (getattr(record, "facets", None) or []):
        for feat in (getattr(facet, "features", None) or []):
            ftype = (
                getattr(feat, "py_type", None)
                or getattr(feat, "type", None)
                or getattr(feat, "$type", None)
            )
            if ftype is None and isinstance(feat, dict):
                ftype = feat.get("$type") or feat.get("type")

            get = (lambda k: feat.get(k)) if isinstance(feat, dict) else (lambda k: getattr(feat, k, None))

            if ftype == "app.bsky.richtext.facet#tag" and get("tag"):
                hashtags.append(get("tag"))
            elif ftype == "app.bsky.richtext.facet#mention" and get("did"):
                mentions.append(get("did"))
            elif ftype == "app.bsky.richtext.facet#link" and get("uri"):
                links.append(get("uri"))

    return {"hashtags": hashtags, "mentions": mentions, "links": links}


## 3. Query design

The pilot revealed which queries are productive and which are not. The query set below applies four changes informed by those results.

- **Queries that saturated the pilot 100-post cap** are scaled up to capture the tail:`"the devil wears prada 2"` is given the highest cap of 1000.

- **High-engagement queries that did not saturate the cap** are kept at 500 posts. This includes the cast queries `"anne hathaway" prada` and `"stanley tucci" prada`, which delivered the highest average likes per post in the pilot, and the press-tour queries `"lady gaga" prada` and `"doechii" prada`.

- **Low-signal queries are removed.** `"the devil wears prada"` (without the "2") returned mostly posts about the 2006 original and is dropped. `"devil wears prada 2"` (without quotation marks framing) returned a small set already covered by the quoted form and is dropped.

- **Six new queries** are added with a conservative limit of 200 each: the abbreviation `"dwp2"` and its hashtag form, two "sequel" phrasings, an explicit year disambiguation, and a movie-context disambiguation.


In [22]:
queries = [
    # Saturated in pilot — high cap
    ('"the devil wears prada 2"',  1000),

    # High signal, did not saturate — full cap
    ('"anne hathaway" prada',       500),
    ('"stanley tucci" prada',       500),
    ('"miranda priestly"',          500),
    ('"anna wintour" prada',        500),
    ("#devilwearsprada2",            500),
    ('"lady gaga" prada',           500),
    ('"doechii" prada',             500),
    ('"emily blunt" prada',         500),

    # Cast member with lower engagement — reduced cap
    ('"meryl streep" prada',        300),

    # New queries — conservative cap
    ('"devil wears prada sequel"',  200),
    ('"prada sequel"',              200),
    ('"dwp2"',                      200),
    ("#dwp2",                        200),
    ('"prada 2" movie',             200),
    ('"devil wears prada" 2026',    200),
]

print(f"{len(queries)} queries, max raw posts before dedup: {sum(n for _, n in queries)}")


16 queries, max raw posts before dedup: 6500


## 4. Time window

The pilot showed that the conversation on Bluesky for this topic does not extend meaningfully before mid-April 2026. The full collection uses the same window as the pilot for end-to-end comparability between the two datasets, with a cautious one-month buffer at the start. The window is closed at 21 May 2026, the date of the pilot collection, even when the full collection is run on a later date: the analysis period reported throughout the project is therefore **1 April 2026 – 21 May 2026**.

In [23]:
SINCE_ISO = "2026-04-01T00:00:00.000Z"
UNTIL_ISO = "2026-05-21T23:59:59.000Z"


## 5. Post collection function

Identical in structure to the pilot collector: cursor-based pagination, server-side time filter with client-side safety check, facet extraction, reply-threading references, and per-query tagging of each row.


In [24]:
def collect_search_posts(
    client,
    q: str,
    limit_total: int = 500,
    page_size: int = 100,
    since_iso: str | None = None,
    until_iso: str | None = None,
    polite_sleep: float = 0.6,
):
    """Collect posts matching `q` with pagination, facets, and reply threading."""
    cursor, rows = None, []
    since_dt = _parse_dt_utc(since_iso) if since_iso else None
    until_dt = _parse_dt_utc(until_iso) if until_iso else None

    while len(rows) < limit_total:
        params = models.AppBskyFeedSearchPosts.Params(
            q=q,
            sort="latest",
            since=since_iso,
            until=until_iso,
            limit=min(page_size, limit_total - len(rows)),
            cursor=cursor,
        )
        res = call_with_retries(client.app.bsky.feed.search_posts, params)
        cursor = res.cursor
        posts = res.posts or []

        if not posts:
            break

        for p in posts:
            rec = getattr(p, "record", None)
            if rec is None:
                continue

            created = getattr(rec, "created_at", None)
            created_dt = _parse_dt_utc(created) if created else None
            if created_dt is not None:
                if since_dt and created_dt < since_dt:
                    continue
                if until_dt and created_dt >= until_dt:
                    continue

            author = getattr(p, "author", None)
            facets = extract_facets(rec)

            reply_obj = getattr(rec, "reply", None)
            reply_root = reply_parent = None
            if reply_obj is not None:
                root_ref = getattr(reply_obj, "root", None)
                parent_ref = getattr(reply_obj, "parent", None)
                reply_root = getattr(root_ref, "uri", None) if root_ref else None
                reply_parent = getattr(parent_ref, "uri", None) if parent_ref else None

            embed = getattr(rec, "embed", None) or getattr(p, "embed", None)
            embed_type = (
                getattr(embed, "py_type", None)
                or getattr(embed, "$type", None)
                or getattr(embed, "type", None)
            )

            labels = []
            for lab in (getattr(p, "labels", None) or getattr(rec, "labels", None) or []):
                val = getattr(lab, "val", None) or (lab.get("val") if isinstance(lab, dict) else None)
                if val:
                    labels.append(val)

            rows.append({
                "query": q,
                "uri": getattr(p, "uri", None),
                "cid": getattr(p, "cid", None),
                "created_at": created_dt,
                "text": getattr(rec, "text", None),
                "author_handle": getattr(author, "handle", None),
                "author_did": getattr(author, "did", None),
                "author_display_name": getattr(author, "display_name", None),
                "reply_count": getattr(p, "reply_count", None),
                "repost_count": getattr(p, "repost_count", None),
                "like_count": getattr(p, "like_count", None),
                "quote_count": getattr(p, "quote_count", None),
                "hashtags": facets["hashtags"],
                "mentions": facets["mentions"],
                "links": facets["links"],
                "langs": getattr(rec, "langs", None),
                "reply_root_uri": reply_root,
                "reply_parent_uri": reply_parent,
                "embed_type": embed_type,
                "labels": labels,
            })

            if len(rows) >= limit_total:
                break

        if cursor is None:
            break
        time.sleep(polite_sleep)

    return rows


## 6.1 Post collection

Each query is collected with its own limit. The progress line for every query reports the number of rows actually returned, which may be lower than the cap if the platform has no more matching posts.


In [25]:
all_rows = []

for q, limit in queries:
    print(f"→ {q}  (cap {limit})")
    rows = collect_search_posts(
        client,
        q=q,
        limit_total=limit,
        page_size=100,
        since_iso=SINCE_ISO,
        until_iso=UNTIL_ISO,
    )
    print(f"  collected: {len(rows)}")
    all_rows.extend(rows)
    time.sleep(1.0)

df_posts_raw = pd.DataFrame(all_rows)
df_posts_raw["created_at"] = pd.to_datetime(df_posts_raw["created_at"], utc=True, errors="coerce")

print(f"\nTotal raw rows collected: {len(df_posts_raw)}")


→ "the devil wears prada 2"  (cap 1000)
  collected: 1000
→ "anne hathaway" prada  (cap 500)
  collected: 500
→ "stanley tucci" prada  (cap 500)
  collected: 345
→ "miranda priestly"  (cap 500)
  collected: 500
→ "anna wintour" prada  (cap 500)
  collected: 194
→ #devilwearsprada2  (cap 500)
  collected: 259
→ "lady gaga" prada  (cap 500)
  collected: 470
→ "doechii" prada  (cap 500)
  collected: 284
→ "emily blunt" prada  (cap 500)
  collected: 443
→ "meryl streep" prada  (cap 300)
  collected: 300
→ "devil wears prada sequel"  (cap 200)
  collected: 108
→ "prada sequel"  (cap 200)
  collected: 130
→ "dwp2"  (cap 200)
  collected: 200
→ #dwp2  (cap 200)
  collected: 17
→ "prada 2" movie  (cap 200)
  collected: 200
→ "devil wears prada" 2026  (cap 200)
  collected: 200

Total raw rows collected: 5150


## 6.2 Deduplication

The sixteen queries overlap by design. Posts are deduplicated on `uri`. The overlap rate offers a first sanity check: a rate similar to the pilot (~22 %) indicates that the queries are converging on the same conversation; a much higher rate would suggest the new queries are redundant, a much lower one that they are catching disjoint subtopics.


In [26]:
df_posts = df_posts_raw.drop_duplicates(subset="uri").reset_index(drop=True)
df_posts = df_posts.sort_values("created_at", ascending=False).reset_index(drop=True)

print(f"Raw rows:        {len(df_posts_raw)}")
print(f"Unique posts:    {len(df_posts)}")
print(f"Unique authors:  {df_posts['author_handle'].nunique()}")
print(f"Overlap rate:    {1 - len(df_posts) / len(df_posts_raw):.1%}")


Raw rows:        5150
Unique posts:    3867
Unique authors:  2407
Overlap rate:    24.9%


In [27]:
# Tag each post as root or reply-from-search; this distinction is preserved
# alongside the replies-from-thread-expansion saved separately in df_replies.
df_posts["is_reply"] = df_posts["reply_parent_uri"].notna()

print()
print(f"Root posts (is_reply=False):           {(~df_posts['is_reply']).sum()}")
print(f"Reply posts from search (is_reply=True):{df_posts['is_reply'].sum()}")


Root posts (is_reply=False):           3487
Reply posts from search (is_reply=True):380


## 6.3 Sanity check

A few descriptive figures before phase 2: the share of post-release activity, the share of English-tagged posts, and the number of root posts available as seeds for thread expansion. The seed set is restricted to root posts with at least one declared reply; reply posts returned by search are not used as seeds (they will be reached as descendants of their respective root posts during thread expansion).

In [28]:
post_release_share = (df_posts["created_at"] >= "2026-05-01").mean()

def _primary_lang(x):
    if isinstance(x, (list, tuple)) and x:
        return x[0]
    return None

df_posts["lang_primary"] = df_posts["langs"].apply(_primary_lang)
en_share = df_posts["lang_primary"].fillna("").str.startswith("en").mean()

root_posts = df_posts[df_posts["reply_parent_uri"].isna()]
seeds = root_posts[root_posts["reply_count"].fillna(0) > 0]

print(f"Post-release share (>= 1 May): {post_release_share:.1%}")
print(f"English-tagged share:          {en_share:.1%}")
print(f"Root posts:                    {len(root_posts)}")
print(f"  └─ with >= 1 declared reply: {len(seeds)}")
print(f"  └─ sum of declared replies:  {int(seeds['reply_count'].sum())}")


Post-release share (>= 1 May): 72.5%
English-tagged share:          60.7%
Root posts:                    3487
  └─ with >= 1 declared reply: 654
  └─ sum of declared replies:  1307


In [29]:
query_summary_full = (
    df_posts.groupby("query")
            .agg(
                posts=("uri", "count"),
                authors=("author_handle", "nunique"),
                median_likes=("like_count", "median"),
                mean_likes=("like_count", "mean"),
                total_replies=("reply_count", "sum"),
            )
            .sort_values("posts", ascending=False)
            .round(2)
)
query_summary_full


,posts,authors,median_likes,mean_likes,total_replies
query,,,,,
"""the devil wears prada 2""",1000,735,0.0,5.14,462
"""miranda priestly""",470,377,0.5,5.56,166
"""anne hathaway"" prada",453,362,0.0,7.00,239
"""lady gaga"" prada",437,263,0.0,5.42,104
"""stanley tucci"" prada",261,187,0.0,9.40,125
#devilwearsprada2,240,179,0.5,2.63,39
"""dwp2""",194,170,1.0,6.04,105
"""anna wintour"" prada",183,146,0.0,3.16,59
"""emily blunt"" prada",171,137,0.0,8.06,75


### Saving posts before thread expansion

The posts dataset and the query-level summary are saved before collecting replies. This creates a checkpoint, so that if the thread expansion loop is interrupted, the already collected posts are not lost. The final save cell writes the same files again to ensure that the latest processed outputs are available.

In [30]:
out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)

def _csv_safe(df: pd.DataFrame, list_cols=()) -> pd.DataFrame:
    out = df.copy()
    for col in list_cols:
        if col in out.columns:
            out[col] = out[col].apply(
                lambda x: "|".join(map(str, x)) if isinstance(x, (list, tuple)) else ""
            )
    return out

posts_list_cols = ("hashtags", "mentions", "links", "langs", "labels")

_csv_safe(df_posts, posts_list_cols).to_csv(out_dir / "bluesky_posts_full.csv", index=False)
df_posts.to_json(
    out_dir / "bluesky_posts_full.jsonl",
    orient="records", lines=True, date_format="iso", force_ascii=False,
)
query_summary_full.to_csv(out_dir / "bluesky_query_summary_full.csv")

print("Phase 1 checkpoint saved:")
print(" -", out_dir / "bluesky_posts_full.csv")
print(" -", out_dir / "bluesky_posts_full.jsonl")
print(" -", out_dir / "bluesky_query_summary_full.csv")

Phase 1 checkpoint saved:
 - ../data/raw/bluesky_posts_full.csv
 - ../data/raw/bluesky_posts_full.jsonl
 - ../data/raw/bluesky_query_summary_full.csv


## 7.1 Thread expansion

For every seed post, the AT Protocol method `app.bsky.feed.get_post_thread` returns a hierarchical view of the conversation rooted at that post. `extract_replies_recursive` walks the returned tree and produces one flat row per reply, recording the URI of the immediate parent and of the thread root, the author, the text, and the same engagement and facet fields used for the posts.

A reply that is itself blocked, deleted or whose author has been suspended is exposed by the API as a `notFound`/`blocked` view rather than a regular `threadViewPost`. These are skipped: there is no author or text to record. The number of skipped nodes is tracked as a coverage statistic.


In [ ]:
def _is_thread_view_post(node) -> bool:
    """Check whether a thread node is a regular ThreadViewPost (not blocked/notFound)."""
    ntype = (
        getattr(node, "py_type", None)
        or getattr(node, "$type", None)
        or getattr(node, "type", None)
    )
    if ntype is None and isinstance(node, dict):
        ntype = node.get("$type") or node.get("type")
    return ntype == "app.bsky.feed.defs#threadViewPost"


def extract_replies_recursive(node, root_uri: str, rows: list, skipped: list):
    """Walk a thread tree and append one row per reply post."""
    replies = getattr(node, "replies", None) or []
    for child in replies:
        if not _is_thread_view_post(child):
            skipped.append(getattr(child, "uri", None))
            continue

        post = getattr(child, "post", None)
        if post is None:
            skipped.append(None)
            continue

        rec = getattr(post, "record", None)
        if rec is None:
            skipped.append(getattr(post, "uri", None))
            continue

        author = getattr(post, "author", None)
        facets = extract_facets(rec)
        created = getattr(rec, "created_at", None)
        created_dt = _parse_dt_utc(created) if created else None

        reply_obj = getattr(rec, "reply", None)
        parent_uri = None
        if reply_obj is not None:
            parent_ref = getattr(reply_obj, "parent", None)
            parent_uri = getattr(parent_ref, "uri", None) if parent_ref else None

        rows.append({
            "uri": getattr(post, "uri", None),
            "cid": getattr(post, "cid", None),
            "root_uri": root_uri,
            "parent_uri": parent_uri,
            "created_at": created_dt,
            "text": getattr(rec, "text", None),
            "author_handle": getattr(author, "handle", None),
            "author_did": getattr(author, "did", None),
            "author_display_name": getattr(author, "display_name", None),
            "reply_count": getattr(post, "reply_count", None),
            "repost_count": getattr(post, "repost_count", None),
            "like_count": getattr(post, "like_count", None),
            "quote_count": getattr(post, "quote_count", None),
            "hashtags": facets["hashtags"],
            "mentions": facets["mentions"],
            "links": facets["links"],
            "langs": getattr(rec, "langs", None),
        })

        # Recurse into deeper replies.
        extract_replies_recursive(child, root_uri, rows, skipped)


In [32]:
def fetch_thread(client, post_uri: str, depth: int = 8):
    """Fetch the full thread for a post URI, returning (rows, n_skipped)."""
    params = models.AppBskyFeedGetPostThread.Params(uri=post_uri, depth=depth)
    res = call_with_retries(client.app.bsky.feed.get_post_thread, params)

    thread = getattr(res, "thread", None)
    if thread is None or not _is_thread_view_post(thread):
        return [], 0

    rows, skipped = [], []
    extract_replies_recursive(thread, root_uri=post_uri, rows=rows, skipped=skipped)
    return rows, len(skipped)


## 7.2 Run thread expansion

The loop processes the seed posts ordered by declared `reply_count` descending, so that the most active threads are fetched first and any rate-limit pauses occur after the most informative threads have been saved. Progress is printed every 25 threads.


In [33]:
seeds_sorted = seeds.sort_values("reply_count", ascending=False).reset_index(drop=True)
seed_uris = seeds_sorted["uri"].tolist()

all_replies = []
total_skipped = 0
threads_with_replies = 0

t0 = time.time()
for i, uri in enumerate(seed_uris, start=1):
    try:
        rows, skipped = fetch_thread(client, uri, depth=8)
    except Exception as e:
        print(f"[{i}/{len(seed_uris)}] failed for {uri}: {e}")
        continue

    total_skipped += skipped
    if rows:
        threads_with_replies += 1
        all_replies.extend(rows)

    if i % 25 == 0 or i == len(seed_uris):
        elapsed = time.time() - t0
        rate = i / elapsed if elapsed > 0 else 0
        print(f"[{i}/{len(seed_uris)}] replies so far: {len(all_replies)} "
              f"| skipped: {total_skipped} | {rate:.1f} threads/s")

    # Checkpoint replies every 50 threads to protect long runs from interruptions.
    if i % 50 == 0 and all_replies:
        pd.DataFrame(all_replies).to_json(
            out_dir / "bluesky_replies_checkpoint.jsonl",
            orient="records", lines=True, date_format="iso", force_ascii=False,
        )

    time.sleep(0.3)

print(f"\nThreads processed:           {len(seed_uris)}")
print(f"Threads with usable replies: {threads_with_replies}")
print(f"Total reply rows extracted:  {len(all_replies)}")
print(f"Skipped (blocked/notFound):  {total_skipped}")

[25/654] replies so far: 572 | skipped: 0 | 1.8 threads/s
[50/654] replies so far: 833 | skipped: 0 | 1.8 threads/s
[75/654] replies so far: 1026 | skipped: 0 | 1.8 threads/s
[100/654] replies so far: 1154 | skipped: 0 | 1.8 threads/s
[125/654] replies so far: 1269 | skipped: 0 | 1.8 threads/s
[150/654] replies so far: 1356 | skipped: 0 | 1.8 threads/s
[175/654] replies so far: 1439 | skipped: 0 | 1.8 threads/s
[200/654] replies so far: 1514 | skipped: 0 | 1.8 threads/s
[225/654] replies so far: 1616 | skipped: 0 | 1.8 threads/s
[250/654] replies so far: 1661 | skipped: 0 | 1.8 threads/s
[275/654] replies so far: 1702 | skipped: 0 | 1.8 threads/s
[300/654] replies so far: 1732 | skipped: 0 | 1.8 threads/s
[325/654] replies so far: 1774 | skipped: 0 | 1.8 threads/s
[350/654] replies so far: 1809 | skipped: 0 | 1.8 threads/s
[375/654] replies so far: 1856 | skipped: 0 | 1.8 threads/s
[400/654] replies so far: 1902 | skipped: 0 | 1.8 threads/s
[425/654] replies so far: 1951 | skipped: 0 |

## 8. Replies DataFrame

Replies are deduplicated on `uri`: the same reply can appear in two threads when one post quotes or branches from another seed, and the API returns it under both. A reply that contains itself replies will appear once as a node (its own URI) and once as a parent reference inside another reply row.


In [34]:
df_replies = pd.DataFrame(all_replies)

if not df_replies.empty:
    df_replies["created_at"] = pd.to_datetime(df_replies["created_at"], utc=True, errors="coerce")
    df_replies = df_replies.drop_duplicates(subset="uri").reset_index(drop=True)
    df_replies = df_replies.sort_values("created_at", ascending=False).reset_index(drop=True)

    # Build a uri → author_handle map from posts + replies, then resolve parent author.
    # This column is what the reply graph in the next notebook needs as edge target.
    uri_to_author = dict(zip(df_posts["uri"], df_posts["author_handle"]))
    uri_to_author.update(dict(zip(df_replies["uri"], df_replies["author_handle"])))
    df_replies["parent_author_handle"] = df_replies["parent_uri"].map(uri_to_author)

print(f"Unique replies:                  {len(df_replies)}")
print(f"Unique reply authors:            {df_replies['author_handle'].nunique() if not df_replies.empty else 0}")
print(f"Replies with resolved parent:    {df_replies['parent_author_handle'].notna().sum() if not df_replies.empty else 0}")
print(f"Replies on >= 1 distinct seed:   {df_replies['root_uri'].nunique() if not df_replies.empty else 0}")

Unique replies:                  2337
Unique reply authors:            1530
Replies with resolved parent:    2337
Replies on >= 1 distinct seed:   644


## 9. Retrieval rate

Two figures characterise how thoroughly the conversations have been collected.
1. The first is the number of replies declared by the seed posts (`sum(reply_count)`), which counts only the *direct* replies to each seed.
2. The second is the number of reply rows actually retrieved through thread expansion with `depth=8`, which includes replies at every depth of each thread — direct replies, replies to replies, and so on down to the eighth level.

The ratio between the two is therefore expected to exceed 100 % whenever the threads contain sub-conversations. A ratio close to 100 % would indicate mostly flat one-level threads; a higher ratio reflects deeper conversational structure. The ratio is reported alongside both raw counts so that the underlying behaviour is unambiguous.

In [37]:
declared_direct = int(seeds["reply_count"].fillna(0).sum())
retrieved_total = len(df_replies)
ratio = retrieved_total / declared_direct if declared_direct else 0.0

# Depth breakdown: how deep do the threads actually go?
if not df_replies.empty:
    # A reply is "first-level" if its parent_uri is the root_uri of its thread.
    first_level = (df_replies["parent_uri"] == df_replies["root_uri"]).sum()
    deeper      = retrieved_total - first_level
else:
    first_level = deeper = 0

print(f"Declared direct replies on seeds: {declared_direct}")
print(f"Retrieved (all depths):           {retrieved_total}")
print(f"  └─ first-level replies:         {first_level}")
print(f"  └─ deeper sub-replies:          {deeper}")
print(f"Ratio retrieved / declared:       {ratio:.1%}")

Declared direct replies on seeds: 1307
Retrieved (all depths):           2337
  └─ first-level replies:         1276
  └─ deeper sub-replies:          1061
Ratio retrieved / declared:       178.8%


## 10. Save

Three deliverables are written under `data/raw/`: the deduplicated posts, the flattened replies, and the per-query summary updated to the full collection. Each is saved both as a CSV with list columns serialised as pipe-separated strings, and as a JSON Lines file that preserves the native list types for direct loading in the next notebook.


In [38]:
out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)

def _csv_safe(df: pd.DataFrame, list_cols=()) -> pd.DataFrame:
    out = df.copy()
    for col in list_cols:
        if col in out.columns:
            out[col] = out[col].apply(
                lambda x: "|".join(map(str, x)) if isinstance(x, (list, tuple)) else ""
            )
    return out

posts_list_cols   = ("hashtags", "mentions", "links", "langs", "labels")
replies_list_cols = ("hashtags", "mentions", "links", "langs")

_csv_safe(df_posts,   posts_list_cols).to_csv(out_dir / "bluesky_posts_full.csv",   index=False)
_csv_safe(df_replies, replies_list_cols).to_csv(out_dir / "bluesky_replies_full.csv", index=False)
query_summary_full.to_csv(out_dir / "bluesky_query_summary_full.csv")

df_posts.to_json(
    out_dir / "bluesky_posts_full.jsonl",
    orient="records", lines=True, date_format="iso", force_ascii=False,
)
df_replies.to_json(
    out_dir / "bluesky_replies_full.jsonl",
    orient="records", lines=True, date_format="iso", force_ascii=False,
)

print("Saved:")
for name in (
    "bluesky_posts_full.csv",
    "bluesky_posts_full.jsonl",
    "bluesky_replies_full.csv",
    "bluesky_replies_full.jsonl",
    "bluesky_query_summary_full.csv",
):
    print(" -", out_dir / name)


Saved:
 - ../data/raw/bluesky_posts_full.csv
 - ../data/raw/bluesky_posts_full.jsonl
 - ../data/raw/bluesky_replies_full.csv
 - ../data/raw/bluesky_replies_full.jsonl
 - ../data/raw/bluesky_query_summary_full.csv


## 11. Summary

The full collection produced two datasets that together form the basis for the rest of the project.

- `posts_full` contains every post matching the sixteen search queries within the period 1 April – 21 May 2026, with engagement metadata, facets, language tags, reply-threading references, and an `is_reply` flag that distinguishes root posts from reply posts returned by search.

- `replies_full` contains every reply reachable through thread expansion of the seed posts, each annotated with the URI of its immediate parent, the URI of the thread root, and the handle of the parent author (resolved against the union of posts and replies). The `parent_author_handle` column is the edge target used by the reply graph in the next notebook.

The next notebook reads these two files and constructs the reply network as a directed graph in which an edge connects the author of a reply to the author of its parent post.